In [0]:
# --- 1. CARICAMENTO MODULI (.py) ---
%run ./engine_utils
%run ./engine_loader
%run ./engine_cleaning
%run ./engine_stats
%run ./engine_config


# --- 2. SELEZIONE CONFIG (Manuale) ---
# Scommenta la riga che ti serve per il test
# config = {260, 288}          # S-WAY
# config = {36, 76, 77}        # EUROCARGO
config = {405}               # X-WAY MY24 AT/AD V1.6.4 C9
# config = {406}             # T-WAY MY24 V1.6.4 C9

# LOGICA DI RICONOSCIMENTO AUTOMATICO
c_list = list(config)
first_c = c_list[0]

# Identificazione gruppo prodotto per logiche condizionali
if first_c in {36, 76, 77}:
    product_group = "Eurocargo"
elif first_c in {399, 400, 401, 402}:
    product_group = "IVECO_S-WAY"
elif first_c == 405:
    product_group = "IVECO_X-WAY"
elif first_c == 406:
    product_group = "IVECO_T-WAY"
else:
    product_group = "IVECO_HEAVY"

# Widget solo per il Reset (comodo per pulire le Delta Table se cambi logica)
dbutils.widgets.dropdown("reset_all", "No", ["Yes", "No"], "Reset totale tabelle?")
force_reset = dbutils.widgets.get("reset_all") == "Yes"

last_running()
print(f"Target: {product_group} | Config: {config}")

In [0]:
p_type, p_group, p_series = extract_metadata(df_clean)
print(f"TEST -> Serie: '{p_series}'")

In [0]:
# --- STAGE 1: RAW ---
table_raw = "stg_fat_raw"
if not spark.catalog.tableExists(table_raw) or force_reset:
    df_raw = import_fat_tables_3(spark, config)
    df_raw.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_raw)
else:
    df_raw = spark.table(table_raw).filter(F.col("id_config").isin(list(config)))
# --- STAGE 2: CLEANED ---
table_clean = "stg_fat_cleaned"
if not spark.catalog.tableExists(table_clean) or force_reset:
    df_clean = clean_spark_column_names(df_raw)
    df_clean = engine_model_standard(df_clean)
    df_clean.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_clean)
else:
    df_clean = spark.table(table_clean).filter(F.col("id_config").isin(list(config)))

report_dim(df_clean, "Dati Pronti per Statistiche")

# --- FIX METADATI ---
# Qui correggiamo l'errore: assegniamo i 3 valori a 3 variabili distinte
p_type, p_group, p_series = extract_metadata(df_clean)

print(f"Analisi per Serie: {bold_start}{p_series}{bold_end}")

# --- SCELTA COLONNE (Logica centralizzata) ---
# Invece di scrivere tutti gli IF/ELIF qui, chiamiamo la funzione di config
sheet_id = "5a"
col_list_5 = get_columns_for_sheet(p_series, p_group, sheet_id)

# Filtriamo per sicurezza quelle realmente presenti nel DF
col_list_5 = [c for c in col_list_5 if c in df_clean.columns]

print(f"Colonne identificate per il foglio {sheet_id}: {col_list_5}")

In [0]:
GROUP_CONFIG

In [0]:
df_clean.display()

In [0]:
# --- CELLA: INIZIALIZZAZIONE EXCEL ---
import pandas as pd
import os

# Definiamo il percorso locale esplicito nella cartella /tmp/
file_output = get_export_file_name(product_group, config)
local_path = f"/tmp/{file_output}" 

# Inizializziamo il writer sul percorso locale
writer = pd.ExcelWriter(local_path, engine='xlsxwriter')

print(f"✅ Writer pronto. Il file verrà creato temporaneamente in: {local_path}")

In [0]:
p_group

In [0]:
p_series

In [0]:
# --- CELLA: FOGLIO 5) CATALYST INFO ---
sheet_id = "5a"

# 1. Estrazione (Valori REALI, non stringhe 'series'!)
p_type, p_group, p_series = extract_metadata(df_clean)

print(f"✅ Dati rilevati dal DB -> Gruppo: {p_group} | Serie: {p_series}")

# 2. Recupero colonne dal dizionario
col_list_raw = get_columns_for_sheet(p_series, p_group, sheet_id)

# 3. Verifica fisica delle colonne
col_list_valid = [c for c in col_list_raw if c in df_clean.columns]
col_missing = [c for c in col_list_raw if c not in df_clean.columns]

if col_missing:
    print(f"⚠️ ATTENZIONE: Queste colonne mancano nella tabella: {col_missing}")

if col_list_valid:
    print(f"📊 Calcolo in corso su: {col_list_valid}")
    df_stats = pyspark_variabili_x(df_clean, col_list_valid, sheet_id)
    
    cols_pivot = [f"{c.upper()}_{sheet_id.upper()}" for c in col_list_valid]
    df_final = report_pivot_pyspark(df_stats, cols_pivot, ["product_group"], [1] * len(cols_pivot))
    
    if df_final is not None:
        display(df_final)
        df_final.to_excel(writer, sheet_name="5) CATALYST INFO", index=False)
else:
    print(f"❌ ERRORE: Non ho trovato nessuna colonna valida per {p_series}")
    print(f"Colonne totali nel DF: {len(df_clean.columns)}")

In [0]:
# --- CELLA: FOGLIO 2b) OIL PRESSURE ---
sheet_id = "2b"

# 1. Recupero la configurazione specifica per questo prodotto e questo foglio
# Se non esiste per questo prodotto, 'conf' sarà None
conf = CONFIG_REPORTS.get(product_group, {}).get(sheet_id)

if conf:
    # 2. Estrazione parametri dal dizionario
    col_list = conf['columns']
    triggers = conf['triggers']
    group_by = conf['group_by']
    sheet_name = conf['name']

    # 3. Controllo di sicurezza: le colonne esistono davvero nel DB?
    # (Evita l'errore UNRESOLVED_COLUMN che abbiamo visto prima)
    valid_cols = [c for c in col_list if c in df_clean.columns]

    if valid_cols:
        print(f"Calcolo {sheet_name} per {product_group}...")
        
        # Esecuzione calcoli
        df_stats = pyspark_variabili_x(df_clean, valid_cols, sheet_id)
        
        # Generazione Pivot (passando solo i trigger per le colonne valide trovate)
        # Usiamo [:len(valid_cols)] per sicurezza nel caso mancasse qualche colonna
        df_final = report_pivot_pyspark(df_stats, valid_cols, group_by, triggers[:len(valid_cols)])
        
        if df_final is not None:
            display(df_final)
            df_final.to_excel(writer, sheet_name=sheet_name, index=False)
            print(f"✅ Foglio '{sheet_name}' salvato correttamente.")
    else:
        print(f"⚠️ Saltato {sheet_name}: Nessuna delle colonne {col_list} trovata nel dataset.")
else:
    print(f"ℹ️ Il foglio {sheet_id} non è previsto per il gruppo {product_group}.")

In [0]:
# --- CELLA FINALE: CHIUSURA E DOWNLOAD ---
import os

# 1. CHIUDIAMO IL WRITER (Fondamentale!)
writer.close()
print("Sincronizzazione file su disco...")

# 2. VERIFICA ESISTENZA
if os.path.exists(local_path):
    print(f"File trovato localmente ({os.path.getsize(local_path)} bytes).")
    
    # 3. CREAZIONE CARTELLA DESTINAZIONE
    dbutils.fs.mkdirs("dbfs:/FileStore/report_gio/")
    
    # 4. COPIA SU DBFS (Usa il prefisso file: per il percorso locale)
    dest_path = f"dbfs:/FileStore/report_gio/{file_output}"
    dbutils.fs.cp(f"file:{local_path}", dest_path)
    
    print("-" * 50)
    print(f"✅ PROCESSO COMPLETATO!")
    print(f"Scarica il file da questo link: https://{spark.conf.get('spark.databricks.workspaceUrl')}/files/report_gio/{file_output}")
    print("-" * 50)
else:
    print(f"❌ ERRORE: Il file {local_path} non è stato creato. Hai eseguito le celle dei fogli?")